In [1]:
import torch
import torch.nn as nn
import torchmetrics
import tokenizers
import transformers 
import random
from collections import namedtuple
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = 'cuda'

/home/damian/New Folder/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
monthes = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
#"April 22, 2019"

def gen_dates_1():
    dates = []
    for year in range(1930, 2027):
        for month in range(0, 12):
            if month == 1:
                if (((year%4) == 0) and not((year%100)==0)) or (year%400)==0:
                    for day in range(1, 30):
                        dates.append(monthes[month] + ' ' + str(day) + ', ' + str(year))
                else:
                    for day in range(1, 29):
                        dates.append(monthes[month] + ' ' + str(day) + ', ' + str(year))
            elif month in [0, 2, 4, 6, 7, 9, 11]:
                for day in range(1, 32):
                    dates.append(monthes[month] + ' ' + str(day) + ', ' + str(year))
            else:
                for day in range(1, 31):
                    dates.append(monthes[month] + ' ' + str(day) + ', ' + str(year))
                    
    return dates

def gen_dates_2():
    dates = []
    month_delimeter = "-"
    day_delimeter = "-"
    for year in range(1930, 2027):
        for month in range(0, 12):
            if month < 9:
                month_delimeter = '-0'
            else:
                month_delimeter = '-'
            if month == 1:
                if (((year%4) == 0) and not((year%100)==0)) or (year%400)==0:
                    for day in range(1, 30):
                        if day < 10:
                            day_delimeter = '-0'
                        else:
                            day_delimeter = '-'
                        dates.append(str(year) +month_delimeter+ str(month+1) +day_delimeter+ str(day))
                else:
                    for day in range(1, 29):
                        if day < 10:
                            day_delimeter = '-0'
                        else:
                            day_delimeter = '-'
                        dates.append(str(year) +month_delimeter+ str(month+1) +day_delimeter+ str(day))
            elif month in [0, 2, 4, 6, 7, 9, 11]:
                for day in range(1, 32):
                    if day < 10:
                        day_delimeter = '-0'
                    else:
                        day_delimeter = '-'
                    dates.append(str(year) +month_delimeter+ str(month+1) +day_delimeter+ str(day))
            else:
                for day in range(1, 31):
                    if day < 10:
                        day_delimeter = '-0'
                    else:
                        day_delimeter = '-'
                    dates.append(str(year) +month_delimeter+ str(month+1) +day_delimeter+ str(day))
                    
    return dates

In [3]:
def gen_dates():
    dates1 = gen_dates_1()
    dates2 = gen_dates_2()
    dates = []
    for i in range(len(dates1)):
        dates.append((dates1[i], dates2[i]))
    random.shuffle(dates)
    return dates

dates = gen_dates()
train_dates = dates[:25000]
valid_dates = dates[25000:30000]
test_dates  = dates[30000:]

train_tokens = []
for date in train_dates:
    train_tokens.append(date[0])
    train_tokens.append(date[1])
    
print(train_tokens[:6])

['December 5, 1968', '1968-12-05', 'August 7, 1976', '1976-08-07', 'August 10, 1944', '1944-08-10']


In [4]:
bpe_tokenizer_model = tokenizers.models.BPE(unk_token='<unk>')
bpe_tokenizer = tokenizers.Tokenizer(bpe_tokenizer_model)
bpe_tokenizer.enable_truncation(max_length=16)
bpe_tokenizer.enable_padding(pad_id=0, pad_token='<pad>')
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bpe_tokenizer_trainer = tokenizers.trainers.BpeTrainer(vocab_size=256, special_tokens=['<pad>', '<unk>', '<s>', '</s>'])
bpe_tokenizer.train_from_iterator(train_tokens, bpe_tokenizer_trainer)

In [5]:
fields = ['src_token_ids', 'src_mask', 'tgt_token_ids', 'tgt_mask']
class Nmt(namedtuple('NmtPairBase',fields)):
    def to(self, device):
        return Nmt(self.src_token_ids.to(device), self.src_mask.to(device),
                   self.tgt_token_ids.to(device), self.tgt_mask.to(device))

def collate_fn(batch):
    src_text = [text[0] for text in batch]
    tgt_text = ['<s>'+text[1]+'</s>' for text in batch]
    src_encodings = bpe_tokenizer.encode_batch(src_text)
    tgt_encodings = bpe_tokenizer.encode_batch(tgt_text)
    src_tokens = torch.tensor([enc.ids for enc in src_encodings])
    tgt_tokens = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = Nmt(src_tokens, src_mask, tgt_tokens[:, :-1], tgt_mask[:, :-1])
    labels = tgt_tokens[:, 1:]
    return inputs, labels

train_loader = DataLoader(train_dates, batch_size=64, collate_fn=collate_fn, shuffle=True)
valid_loader = DataLoader(valid_dates, batch_size=64, collate_fn=collate_fn)
test_loader  = DataLoader(test_dates, batch_size=64, collate_fn=collate_fn)

In [6]:
def eval_model(model, metric, valid_loader):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
        
        return metric.compute()
    
def train_model(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs, patience=10, factor=0.1):
    history = {'train_loss':[], 'train_metric':[], 'valid_metric':[]}
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, mode='min', factor=factor, patience=patience)
    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        
        history['train_loss'].append(total_loss/len(train_loader))
        history['train_metric'].append(metric.compute().item())
        val_metric = eval_model(model, metric, valid_loader).item()
        history['valid_metric'].append(val_metric)
        scheduler.step(val_metric)
        print(f'Epoch: {epoch+1}\tTrain Loss: {history["train_loss"][-1]:.3f}\tTrain Metric: {history["train_metric"][-1]:.3f}\tValid Metric: {history['valid_metric'][-1]:.3f}')
        
    return history

In [7]:
def attention(query, key, value):
    score = query @ key.transpose(1, 2)
    weights = torch.softmax(score, dim=-1)
    return weights @ value

class DateModel(nn.Module):
    def __init__(self, vocab_size, embed_size=10, hidden_size=128, n_layers=2, pad_id=0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=pad_id)
        self.encoder = nn.GRU(embed_size, hidden_size, n_layers, batch_first=True)
        self.decoder = nn.GRU(embed_size, hidden_size, n_layers, batch_first=True)
        self.output = nn.Linear(hidden_size*2, vocab_size)
        
    def forward(self, X):
        src_embeddings = self.embed(X.src_token_ids)
        tgt_embeddings = self.embed(X.tgt_token_ids)
        src_length = X.src_mask.sum(dim=1)
        src_packed = pack_padded_sequence(src_embeddings, lengths=src_length.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden_state = self.encoder(src_packed)
        decoder_outputs, _ = self.decoder(tgt_embeddings, hidden_state)
        encoder_outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        attention_output = attention(decoder_outputs, encoder_outputs, encoder_outputs)
        combined_output = torch.cat((attention_output, decoder_outputs), dim=-1)
        return self.output(combined_output).permute(0, 2, 1)

In [9]:
model_1 = DateModel(vocab_size=256).to(device)

optimizer = torch.optim.NAdam(params=model_1.parameters())
xentropy = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task='multiclass', num_classes=256).to(device)

history = train_model(model_1, optimizer, xentropy, metric, train_loader, valid_loader, 10)

Epoch: 1	Train Loss: 1.067	Train Metric: 0.746	Valid Metric: 0.916
Epoch: 2	Train Loss: 0.080	Train Metric: 0.979	Valid Metric: 1.000
Epoch: 3	Train Loss: 0.005	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 4	Train Loss: 0.002	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 5	Train Loss: 0.001	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 6	Train Loss: 0.001	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 7	Train Loss: 0.000	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 8	Train Loss: 0.000	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 9	Train Loss: 0.000	Train Metric: 1.000	Valid Metric: 1.000
Epoch: 10	Train Loss: 0.000	Train Metric: 1.000	Valid Metric: 1.000


In [64]:
eval_model(model_1, metric, test_loader)

tensor(1., device='cuda:0')

In [63]:
X = [date[0] for date in test_dates[:10]]
print(X)

def format_date(model, date):
    tgt_text = ""
    model.eval()
    for i in range(20):
        batch, _ = collate_fn([(date, tgt_text)])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_id = Y_logits.argmax(dim=1)
            next_token_id = Y_token_id[0, i]
            
        next_token = bpe_tokenizer.id_to_token(next_token_id)
        if 'Ġ' in next_token:
            next_token = next_token[1:]
        tgt_text += next_token
        if next_token_id == 3:
            break
    return tgt_text

for date in X:
    print(format_date(model_1, date))

['September 8, 2004', 'April 22, 1994', 'May 31, 2015', 'August 13, 2010', 'January 10, 1954', 'September 28, 1997', 'February 4, 1969', 'January 23, 2022', 'September 15, 2005', 'December 9, 1950']
2004-09-08</s>
1994-04-22</s>
2015-05-31</s>
2010-08-13</s>
1954-01-10</s>
1997-09-28</s>
1969-02-04</s>
2022-01-23</s>
2005-09-15</s>
1950-12-09</s>
